# Retrain T1.4 / T1.4b / T1.4c / T1.5 smoke: multi-corpus on BEIR

Validates four stacked features end-to-end on SciFact + NFCorpus + FiQA:

- **T1.4** -- `retrain_multi()` with temperature sampling.
- **T1.4b** -- `bulk_mine=True` replaces per-query `store.search` in triple
  generation with one GPU matmul. Mining time: ~3 h -> ~2 min on FiQA.
- **T1.4c** -- `bulk_eval=True` applies the same trick to baseline + final
  `evaluate_model`. Eval time on 3 corpora: ~60 min -> ~5 min.
- **T1.5 (new)** -- `training_queries_by_dataset` routes triple mining
  through the v5 recipe: real BEIR `queries.jsonl` text + gold doc first
  chunks as positives + multiple hard negs per query. This is the recipe
  that produced `Stffens/bge-small-rrf-v2` (+7.3% SciFact, +18.9% NFCorpus).
  Chunk-prefix training (the T1.4 default) damaged SciFact/FiQA -5/-9% on
  the 2026-04-18 run because statements-as-queries do not match BEIR's
  question-based evaluation.

**Recipe comparison:**

| | Chunk-prefix (T1.4) | Labeled (T1.5) |
|---|---|---|
| Training query | `chunk[:200]` | Real `queries.jsonl` |
| Positive | The source chunk | Gold doc's first chunk |
| Hard negs per query | 1 | All vec_only + fts_only |
| RRF weights | Adaptive by length | Fixed 0.95/0.05 |

**Regression target**: per-dataset NDCG@10 deltas must match the v5 notebook
within noise (NFCorpus > +10%, SciFact > +2%). Macro delta positive.

Runtime on T4 with all four features: ~25-35 min end-to-end.

In [ ]:
# Cell 1: Setup -- vstash develop (T1.4 + T1.4b already merged).
!pip install -q 'sentence-transformers>=3' torch 'accelerate>=1.1.0'
!rm -rf /content/vstash
!git clone --branch develop https://github.com/stffns/vstash.git /content/vstash
%cd /content/vstash
!pip install -q -e .

In [ ]:
# Cell 2: Download each BEIR dataset and ingest into a separate vstash store.
# download_beir uses experiments/data relative to cwd, which is gitignored --
# a fresh clone does not have it. Create it before importing.
import os
import shutil
import sys

os.chdir("/content/vstash")
sys.path.insert(0, "/content/vstash")
os.makedirs("experiments/data", exist_ok=True)

from experiments.beir_benchmark import download_beir, load_beir
from sentence_transformers import SentenceTransformer
from vstash.store import VstashStore

BASE_MODEL = "BAAI/bge-small-en-v1.5"

# NFCorpus is small (~3.6k), SciFact mid (~5k), FiQA large (~57k). Good mix for
# temperature sampling to show its effect.
DATASETS = ["scifact", "nfcorpus", "fiqa"]

store_paths = {name: f"/tmp/retrain_t14_{name}.db" for name in DATASETS}
OUTPUT_PATH = "/content/retrained_model_t14_multi"

for p in [
    *list(store_paths.values()),
    OUTPUT_PATH,
    OUTPUT_PATH + ".candidate",
    OUTPUT_PATH + ".old",
]:
    for suffix in ("", "-wal", "-shm"):
        target = p + suffix
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.isfile(target):
            os.remove(target)

model = SentenceTransformer(BASE_MODEL)

per_dataset = {}
stores = {}
for name in DATASETS:
    cache = download_beir(name)
    corpus, queries, qrels = load_beir(cache)
    doc_ids = list(corpus.keys())
    print(f"[{name}] corpus: {len(doc_ids)} docs | queries: {len(queries)} | qrels: {len(qrels)}")
    per_dataset[name] = {"corpus": corpus, "queries": queries, "qrels": qrels}

    texts = [
        (corpus[d].get("title", "") + " " + corpus[d].get("text", "")).strip() for d in doc_ids
    ]
    vecs = model.encode(texts, normalize_embeddings=True, show_progress_bar=True, batch_size=128)
    store = VstashStore(store_paths[name], embedding_dim=int(vecs.shape[1]))
    for doc_id, text, vec in zip(doc_ids, texts, vecs):
        store.add_document(
            path=f"{name}://{doc_id}",
            title=corpus[doc_id].get("title", "")[:80] or doc_id,
            chunks=[text],
            embeddings=[list(map(float, vec))],
        )
    stats = store.stats()
    print(f"  -> ingested {stats.documents} docs / {stats.chunks} chunks")
    stores[name] = store

In [ ]:
# Cell 3: Build per-dataset eval sets from real qrels.
from vstash.retrain import qrels_to_eval_queries

eval_queries_by_dataset = {}
for name, bundle in per_dataset.items():
    eval_queries = qrels_to_eval_queries(
        queries=bundle["queries"],
        qrels=bundle["qrels"],
        path_for_doc_id=lambda doc_id, d=name: f"{d}://{doc_id}",
    )
    eval_queries_by_dataset[name] = eval_queries
    print(f"[{name}] eval_queries (real qrels): {len(eval_queries)}")

In [ ]:
# Cell 4: Inspect the triple budget before training so we catch sampling bugs
# without paying for training first.
from vstash.retrain import compute_triple_budget

TOTAL_TRIPLES = 30000
SAMPLING = "temperature"
TEMPERATURE = 0.5

sizes = {name: s.stats().chunks for name, s in stores.items()}
budget = compute_triple_budget(
    sizes,
    total_triples=TOTAL_TRIPLES,
    strategy=SAMPLING,
    temperature=TEMPERATURE,
)
print(f"sampling={SAMPLING} temperature={TEMPERATURE} total_triples={TOTAL_TRIPLES}")
for name in DATASETS:
    frac = budget[name] / TOTAL_TRIPLES
    raw_frac = sizes[name] / sum(sizes.values())
    print(
        f"  {name:<10} chunks={sizes[name]:>6}  raw_frac={raw_frac:.2%}  "
        f"budget={budget[name]:>6}  eff_frac={frac:.2%}"
    )

In [ ]:
# Cell 5: Run retrain_multi with the v5 recipe.
# T1.5 key change: training_queries_by_dataset = eval_queries_by_dataset.
# This feeds BEIR's real queries.jsonl as training queries and uses each
# gold doc's first chunk as the positive (matching v5's recipe that
# produced Stffens/bge-small-rrf-v2). Multiple triples emitted per query
# (one per gold x hard_neg). Chunk-prefix training is the wrong recipe
# for question-based datasets -- it damaged SciFact/FiQA by -5/-9% on
# 2026-04-18's first run.
import time
from vstash.retrain import retrain_multi

EVAL_NOISE = max(max(sizes.values()), 10000)

t0 = time.perf_counter()
result = retrain_multi(
    stores,
    base_model=BASE_MODEL,
    output_path=OUTPUT_PATH,
    sampling=SAMPLING,
    temperature=TEMPERATURE,
    total_triples=TOTAL_TRIPLES,
    epochs=2,
    lr=3e-6,
    batch_size=32,
    use_amp=True,
    max_seq_length=256,
    bulk_mine=True,
    bulk_eval=True,
    training_queries_by_dataset=eval_queries_by_dataset,
    eval_queries_by_dataset=eval_queries_by_dataset,
    eval_noise_size=EVAL_NOISE,
    min_gain=0.0,
    per_dataset_gate=False,
)
elapsed = time.perf_counter() - t0
print(f"\nretrain_multi finished in {elapsed:.1f}s")

In [ ]:
# Cell 6: Pretty-printed per-dataset + macro report.
import json
from pathlib import Path


def fmt(m):
    if m is None or m.n_queries == 0:
        return "(no queries)"
    return f"NDCG@10={m.ndcg_at_10:.4f}  MRR={m.mrr:.4f}  Hit@10={m.hit_at_10:.4f}  n={m.n_queries}"


print("=" * 70)
print(f"T1.4 multi-corpus training: sampling={result.sampling} temperature={result.temperature}")
print("=" * 70)
print(f"total_pairs:        {result.total_pairs}")
print(f"gated_out:          {result.gated_out}")
print(f"per_dataset_gate:   {result.per_dataset_gate}")
print(f"min_gain:           {result.min_gain:+.4f}")
print(f"output_path:        {result.output_path}")
print()
print("Per-dataset budget / pairs emitted:")
for name in DATASETS:
    print(
        f"  {name:<10} budget={result.per_dataset_budget.get(name, 0):>6}  "
        f"emitted={result.per_dataset_pairs.get(name, 0):>6}"
    )
print()
print("Per-dataset eval (baseline -> final):")
for name in DATASETS:
    b = result.per_dataset_baseline.get(name)
    f = result.per_dataset_final.get(name)
    delta = result.per_dataset_delta.get(name, 0.0)
    print(f"  {name}:")
    print(f"    baseline: {fmt(b)}")
    print(f"    final:    {fmt(f)}")
    print(f"    delta NDCG@10: {delta:+.4f}  ({delta * 100:+.2f}%)")
print()
print(f"macro baseline NDCG@10: {result.macro_baseline_ndcg:.4f}")
print(f"macro final NDCG@10:    {result.macro_final_ndcg:.4f}")
print(
    f"macro delta NDCG@10:    {result.macro_delta_ndcg:+.4f}  ({result.macro_delta_ndcg * 100:+.2f}%)"
)

meta_path = Path(result.output_path or (OUTPUT_PATH + ".candidate")) / "training_meta.json"
if meta_path.exists():
    print()
    print("training_meta.json (multi_eval block):")
    meta = json.loads(meta_path.read_text())
    print(json.dumps(meta.get("multi_eval", {}), indent=2))